In [1]:
import pandas as pd
import sqlite3

master = pd.read_csv('../data/clean/master.csv')
conn = sqlite3.connect('../data/olist.db')
master.to_sql('master', conn, if_exists='replace', index=False)

pd.read_sql('SELECT COUNT(*) AS n_rows FROM master', conn)

,n_rows
0,109872


In [2]:
query = '''
WITH monthly AS (
  SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
         COUNT(DISTINCT order_id) AS orders,
         SUM(price) AS revenue
  FROM master
  GROUP BY month
)
SELECT month, orders, ROUND(revenue, 0) AS revenue,
       ROUND(revenue / orders, 2) AS aov,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS mom_pct
FROM monthly
ORDER BY month;
'''
pd.read_sql(query, conn)

,month,orders,revenue,aov,mom_pct
0,2017-01,750,111798.0,149.06,NaN
1,2017-02,1653,234223.0,141.70,109.5
2,2017-03,2546,359199.0,141.08,53.4
3,2017-04,2303,340670.0,147.92,-5.2
4,2017-05,3545,489159.0,137.99,43.6
5,2017-06,3135,421923.0,134.58,-13.7
6,2017-07,3872,481605.0,124.38,14.1
7,2017-08,4193,554700.0,132.29,15.2
8,2017-09,4150,607400.0,146.36,9.5
9,2017-10,4478,648248.0,144.76,6.7
